# Topical Map Engine — v1 Full Pipeline (Stages 1-8)

Runs the complete 8-stage pipeline end-to-end and produces a downloadable Markdown report + JSON.

## Setup

In [ ]:
!pip install -q pydantic anthropic jinja2

In [ ]:
# Load the Anthropic API key from Colab Secrets (with fallback to manual entry).
import os

if not os.environ.get('ANTHROPIC_API_KEY'):
    try:
        from google.colab import userdata
        os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
        print('Loaded ANTHROPIC_API_KEY from Colab Secrets.')
    except Exception as e:
        from getpass import getpass
        print(f'Colab Secrets not available ({type(e).__name__}). Falling back to manual entry.')
        os.environ['ANTHROPIC_API_KEY'] = getpass('Anthropic API key: ')
else:
    print('ANTHROPIC_API_KEY already set in environment.')

In [ ]:
# Robust project setup: unzip if needed, locate engine root, set up imports.
import os, sys, glob, subprocess

zip_files = glob.glob('/content/*.zip')
if zip_files and not os.path.isdir('/content/topical_map_engine'):
    os.system(f'unzip -o -q "{zip_files[0]}" -d /content/')

result = subprocess.run(
    ['find', '/content', '-name', 'models.py', '-not', '-path', '*/sample_data/*'],
    capture_output=True, text=True,
)
found = [p for p in result.stdout.strip().split('\n') if p]
if not found:
    raise FileNotFoundError('Engine folder not found. Upload the zip via the Files panel.')

PROJECT_ROOT = os.path.dirname(found[0])
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

for k in list(sys.modules.keys()):
    if k == 'models' or k == 'stages' or k.startswith('stages.') or k == 'pipeline':
        del sys.modules[k]

print(f'Working directory: {os.getcwd()}')

import models
from stages.intake import load_intake_from_json
from pipeline import run_pipeline
print('All imports successful.')

## Run the Full Pipeline

Expected runtime: 4-7 minutes. Cost: roughly $0.50-1.00 in Anthropic spend.

If you want to skip stage 4 (web validation) for faster iteration, set `skip_validation=True`.

In [ ]:
seed = load_intake_from_json('examples/wordpress_dev/input.json')

output = run_pipeline(
    seed=seed,
    output_dir='examples/wordpress_dev/output',
    skip_validation=False,
)

## Inspect Results

In [ ]:
tm = output.topical_map
print(f'Central entity: {tm.central_entity.primary}')
print(f'Pillars: {len(tm.pillars)}')
print(f'Total clusters: {sum(len(p.clusters) for p in tm.pillars)}')
print(f'Total supplementary nodes: {sum(sum(len(c.supplementary_nodes) for c in p.clusters) for p in tm.pillars)}')
total_queries = sum(
    len(p.representative_queries) + sum(len(c.represented_queries) for c in p.clusters)
    for p in tm.pillars
)
print(f'Total queries: {total_queries}')
print(f'Geo pages: {len(tm.geo_pages)}')
print(f'Internal links: {len(output.linking_plan.links)}')
print(f'Homepage links: {len(output.linking_plan.homepage_links)}')

In [ ]:
from IPython.display import Markdown, display

with open('examples/wordpress_dev/output/topical_map_report.md') as f:
    display(Markdown(f.read()))

In [ ]:
try:
    from google.colab import files
    files.download('examples/wordpress_dev/output/topical_map_report.md')
    files.download('examples/wordpress_dev/output/topical_map.json')
except ImportError:
    print('Not running in Colab — files are at examples/wordpress_dev/output/')